# 06 · Brief C board answer · Group 10

**Produces:** the H1 2026 invoice-to-capture bridge and one board-ready sentence with amounts, counts, timing, and a Finance action.

**Project sequence:** active scheduled test volume grew 4.97% (rounds to 5%) against 18.32% implied plan growth. After data cleaning and exploratory Brief A/B work, Group 10 selected Brief C for the COO's finance question.

Run this notebook from the bundled data pack. It cleans only finance tables, preserves local source provenance, reconciles invoices to captured attempts through 30 June, and observes later captures through 15 July. Missing matches are investigation items, not proof of permanent loss.

## CONFIG

In [ ]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CONFIG = {
    "brief": "Brief C", "group": 10,
    "data_pack": ROOT / "data" / "Build1_DataPack.zip",
    "clean": ROOT / "data" / "clean" / "brief_c",
    "output": ROOT / "outputs" / "brief_c",
    "action": ROOT / "data" / "clean" / "brief_c" / "actions",
    "year": 2026, "observation_cutoff": "2026-07-15",
}
print(f"Kalpa Health · {CONFIG['brief']} · Group {CONFIG['group']}")

In [ ]:
import json
import subprocess
import sys
from decimal import Decimal

subprocess.run([sys.executable, str(ROOT / "clean_brief_c.py"), "--source-dir", str(CONFIG["data_pack"]), "--output-dir", str(CONFIG["clean"])], cwd=ROOT, check=True)
subprocess.run([sys.executable, str(ROOT / "analyze_brief_c_reconciliation.py"), "--data-dir", str(CONFIG["clean"]), "--output-dir", str(CONFIG["output"]), "--action-dir", str(CONFIG["action"]), "--year", str(CONFIG["year"]), "--observation-cutoff", CONFIG["observation_cutoff"]], cwd=ROOT, check=True)
summary = json.loads((CONFIG["output"] / "finance_bridge_summary.json").read_text())
cohort, attempts = summary["invoice_cohort"], summary["payment_attempts_created_in_h1"]
at_close_count = cohort["invoices_with_no_capture_by_h1_end"] + cohort["partially_paid_invoices_by_h1_end"]
crore = lambda value: f"₹{Decimal(value) / Decimal(10_000_000):.3f} crore"
HEADLINE = {
    "number": Decimal(cohort["positive_invoice_balance_rs_by_h1_end"]) / Decimal(10_000_000),
    "numerator": at_close_count, "denominator": cohort["invoice_count"],
}
open_pct = Decimal(cohort["remaining_positive_invoice_balance_rs_by_observation_cutoff"]) / Decimal(cohort["invoice_amount_rs"]) * 100
HEADLINE["claim"] = f"At H1 2026 close, {at_close_count:,} of {cohort['invoice_count']:,} invoices ({crore(cohort['invoice_amount_rs'])} billed) carried {crore(cohort['positive_invoice_balance_rs_by_h1_end'])} in positive balances; of {cohort['invoices_with_no_capture_by_h1_end']:,} with no captured match, {cohort['no_capture_invoices_with_capture_by_observation_cutoff']:,} received later captures ({crore(cohort['face_amount_rs_of_those_later_paid_invoices'])} face value), leaving {crore(cohort['remaining_positive_invoice_balance_rs_by_observation_cutoff'])} ({open_pct:.2f}% of H1 billings) open across {cohort['remaining_positive_balance_invoice_count_by_observation_cutoff']:,} invoices by 15 July, while {attempts['unmatched_capture_rows']:,} H1 captured attempts worth {crore(attempts['unmatched_capture_amount_rs'])} remained unmatched—Finance should prioritize June's open invoices and resolve both exception lists before calling balances lost."
print(HEADLINE["claim"])
print(f"H1 positive balance: {HEADLINE['number']:.3f} crore ({cohort['positive_invoice_balance_rs_by_h1_end']} rupees)")
print(f"Open balance as a share of H1 billing: {open_pct:.2f}%")

## The board sentence and its audit trail

In [ ]:
import pandas as pd
bridge = pd.read_csv(CONFIG["output"] / "finance_bridge_summary.csv", dtype=str)
print(bridge.to_string(index=False))
print("\nExact amounts, matching coverage and caveats: outputs/brief_c/finance_bridge_summary.json")

## Reconciliation checks

In [ ]:
assert HEADLINE["numerator"] <= HEADLINE["denominator"]
assert Decimal(cohort["positive_invoice_balance_rs_by_h1_end"]) == Decimal(cohort["no_capture_invoice_amount_rs_by_h1_end"]) + Decimal(cohort["partial_unpaid_balance_rs"])
assert "Finance should investigate" in HEADLINE["claim"]
print(f"PASS · {CONFIG['brief']} · Group {CONFIG['group']} · H1 {CONFIG['year']}")
print("PASS · open balances, later captures and unmatched attempts are reported separately")

The board sentence above is sourced from the generated JSON summary. Finance should confirm source completeness and bank/cash coverage before treating any balance as a loss.

## Board-question answers

- **Why not call open invoices lost?** Captures can arrive after close, references can be malformed, and later reversals/source gaps have not been ruled out.
- **Why keep overcaptures separate?** The 704 overcaptured invoices (₹0.117 crore) do not establish that credits may be applied to another customer's invoice.
- **Is the settlement gap a bank shortfall?** No. H1 settlements net to ₹24.537 crore, and linked captures span payment dates; independent bank statements and cash registers were not supplied.
- **Where should Finance start?** June invoices account for 45.8% of the 15 July open balance; review that cohort and the two local action lists first. This is a priority, not a causal finding.